# dim_playstyle.py

## Purpose
Builds `dim_playstyle` as an SCD2 dimension that tracks the evolution of the EA FC PlayStyles catalog.

## Source
- Bronze: `fifa.bronze.ea_fc_players_raw` (column `playstyles`, comma-separated)

## Business Rules
- Each PlayStyle can be base or plus ("Finesse Shot" vs "Finesse Shot+")
- They are two distinct records in the dimension
- Business key: `playstyle_code` + `is_plus`
- Surrogate key: `xxhash64(playstyle_raw)` — includes the `+` if applicable

## Strategy
- SCD2 with MERGE
- Expires the current record when `playstyle_name` changes

## Output
- Table: `fifa.silver.dim_playstyle`

## Columns
- `playstyle_id` (INT)
- `playstyle_code`, `playstyle_name` (STRING)
- `is_plus` (BOOLEAN)
- `Start_date`, `End_date`, `Is_current` (SCD2 control columns)
- `_ingestion_timestamp` (TIMESTAMP)

## Why SCD2?

The PlayStyles catalog evolves across editions:

- New PlayStyles are added (e.g. "Gamechanger" introduced in FC 24).
- Some PlayStyles are renamed or removed.
- Base and plus versions are tracked separately.

SCD2 keeps the full history of these changes, so you can see which PlayStyles existed in each snapshot and how they evolved.

## Business key

The business key is `playstyle_code + is_plus`:

- `"Finesse Shot"` (base) and `"Finesse Shot+"` (plus) are two distinct records.
- The `playstyle_id` is a hash of `playstyle_raw` (with `+`), so base and plus get different IDs.

## Parsing the playstyles column

The `playstyles` column in Bronze is a comma-separated string (e.g. `"Finesse Shot+, Technical, First Touch"`). It's exploded into rows, trimmed, and split into:

- `playstyle_name`: the name without the `+`.
- `is_plus`: whether the PlayStyle is a plus version.
- `playstyle_raw`: the original value with the `+` (used for hashing).



In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64
from delta.tables import DeltaTable

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

# Explode playstyles: split by comma, trim, filter NULLs
playstyles_exploded = (
    bronze
    .select(col("playstyles"))
    .filter(col("playstyles").isNotNull() & (col("playstyles") != ""))
    .select(explode(split(col("playstyles"), ",")).alias("playstyle_raw"))
    .select(trim(col("playstyle_raw")).alias("playstyle_raw"))
    .filter(col("playstyle_raw") != "")
    .distinct()
    # Detect if the PlayStyle is a plus version (ends with "+")
    .withColumn("is_plus",
        when(col("playstyle_raw").endswith("+"), True).otherwise(False)
    )
    # Extract the name without the trailing "+"
    # Example: "Finesse Shot+" -> "Finesse Shot"
    .withColumn("playstyle_name",
        regexp_extract(col("playstyle_raw"), r"^(.+?)\+?$", 1)
    )
    .select("playstyle_raw", "playstyle_name", "is_plus")  # ← conserva raw
    .distinct()
)

# Build the SCD2-ready DataFrame
new_playstyles = (
    playstyles_exploded
    # Hash the raw value (with "+") so base and plus get different IDs
    .withColumn("playstyle_id", pmod(xxhash64(col("playstyle_raw")), 1000000).cast("int"))
    .withColumn("playstyle_code", col("playstyle_name"))
    .withColumn("Start_date", current_timestamp())
    .withColumn("End_date", lit(None).cast("timestamp"))
    .withColumn("Is_current", lit(True))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select(
        "playstyle_id", "playstyle_code", "playstyle_name", "is_plus",
        "Start_date", "End_date", "Is_current", "_ingestion_timestamp"
    )
)


# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_playstyle (
    playstyle_id INT,
    playstyle_code STRING,
    playstyle_name STRING,
    is_plus BOOLEAN,
    Start_date TIMESTAMP,
    End_date TIMESTAMP,
    Is_current BOOLEAN,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# MERGE SCD2

target = DeltaTable.forName(spark, "fifa.silver.dim_playstyle")

(
    target.alias("t")
    .merge(
        new_playstyles.alias("s"),
        "t.playstyle_code = s.playstyle_code AND t.is_plus = s.is_plus AND t.Is_current = true"
    )
    # If the name changed, expire the current record
    .whenMatchedUpdate(
        condition="t.playstyle_name <> s.playstyle_name",
        set={
            "End_date": "s.Start_date",
            "Is_current": "false"
        }
    )
    # Insert new records (or new versions of changed records)
    .whenNotMatchedInsert(
        values={
            "playstyle_id": "s.playstyle_id",
            "playstyle_code": "s.playstyle_code",
            "playstyle_name": "s.playstyle_name",
            "is_plus": "s.is_plus",
            "Start_date": "s.Start_date",
            "End_date": "s.End_date",
            "Is_current": "s.Is_current",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .execute()
)

print(f"dim_playstyle actualizada con {spark.read.table('fifa.silver.dim_playstyle').count()} filas")

LIMIT 10 of the dim_playstyle table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_playstyle ORDER BY playstyle_name LIMIT 10;